In [ ]:
import MDAnalysis as mda
import pandas as pd
import numpy as np
from ast import literal_eval
from mg_sampling import PROJECT_DIR

In [ ]:
converters = {f'{a}_{b}': literal_eval for a in ('inner', 'outer') for b in ('indices', 'names', 'resids', 'resnames', 'dist')}

df_pdb = pd.read_csv(PROJECT_DIR / 'structures/7ez0/mMg_tip3p/ion_motifs.csv', index_col=0, converters=converters).drop(columns=['frame', 'time']).explode([f'inner_{a}' for a in ('indices', 'names', 'resids', 'resnames', 'dist')])
df_eq = pd.read_csv(PROJECT_DIR / 'structures/7ez0/mMg_tip3p/rep00/ion_motifs.csv', index_col=0, converters=converters).drop(columns=['frame', 'time']).explode([f'inner_{a}' for a in ('indices', 'names', 'resids', 'resnames', 'dist')])

In [ ]:
df_pdb.head()

In [ ]:
df = pd.merge(df_pdb, df_eq, on=['index', 'name', 'inner_indices', 'inner_names', 'inner_resids', 'inner_resnames'], suffixes=('_pdb', '_eq'), how='outer')
df['pdb_resid'] = df['index'] - 12442 + 501

In [ ]:
df.head()

In [ ]:
def get_outer_inner_dist(df):
    outer_indices = df['outer_indices_pdb']
    if isinstance(outer_indices, list):
        inner_index = df['inner_indices']
        if inner_index in outer_indices:
            i = outer_indices.index(inner_index)
            return df['outer_dist_pdb'][i]
        else:
            return np.nan
    else:
        return np.nan

df['outer_dist'] = df.apply(get_outer_inner_dist, axis=1)

In [ ]:
df

In [ ]:
df['outer_to_inner'] = df.apply(lambda x: x['inner_indices'] if x['inner_indices'] in x['outer_indices_pdb'] else np.nan)

In [ ]:
df

In [ ]:
def format_mgrna(s):
    mapping = {
        'Oph': r'$\mathrm{O_{ph}}$',
        'Ob': r'$\mathrm{O_b}$',
        'fac': r'\textit{fac}',
        'cis': r'\textit{cis}',
        'Cis': r'\underline{\textit{Cis}}',
    }
    for k, v in mapping.items():
        s = s.replace(k, v)
    return s

header = ['Res.', 'Inner atoms', '$d$ [Å]', 'MgRNA class', 'Inner atoms', '$d$ [Å]', 'MgRNA class']
mask = (df['inner_labels_pdb'] != "") | (df['inner_labels_eq'] != "")
print(
    df[['pdb_resid', 'inner_labels_pdb', 'inner_dist_pdb', 'mgrna_pdb', 'inner_labels_eq', 'inner_dist_eq', 'mgrna_eq']].loc[mask].replace("", "--").to_latex(
        index=False,
        header=header,
        column_format='l|XXl|XXl',
        na_rep="--",
        formatters={
            'pdb_resid': lambda x: f"\hline {x}",
            'mgrna_pdb': format_mgrna,
            'mgrna_eq': format_mgrna,
        },
    ).replace(r'\begin{tabular}', r'\begin{tabularx}{\linewidth}').replace(r'\end{tabular}', r'\end{tabularx}').replace(r'\toprule', r'\toprule & \multicolumn{3}{c|}{\textbf{PDB}} & \multicolumn{3}{c}{\textbf{After NPT equilibration}} \\')
)

In [ ]:
mask